# 11 – Dinamikus kockázat és intelligens risk-based maintenance

**12. tanegység · javasolt idő: 2 × 50 perc**

## Tanulási célok

- statikus és dinamikus risk estimate elkülönítése
- szenzorevidencia time-dependent probability-vá alakítása
- dynamic FTA risk trend számítása
- maintenance now/wait trigger expected-cost alapján

**PUNDIT kapcsolat.** Ez a tanegység a PUNDIT projekt (`2020-1.2.3-EUREKA-2022-00021`) eredményeinek **oktatási hasznosítását** támogatja; önmagában nem helyettesíti és nem igazolja a projekt eredeti műszaki vállalásainak teljesítését.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Projekt gyökér:", ROOT)


## 1. Statikus prior → dinamikus risk

A 3. notebookban egyetlen $P(OVERFLOW)$ számot kaptunk. Üzem közben az evidencia változik, ezért $P(H|E_{0:t})$ időfüggő. A digitálisiker-alapú döntéstámogatás egyik lényege, hogy a modellállapot és evidence rendszeresen frissüljön.


In [ ]:
df=pd.read_csv(ROOT/'data'/'pundit_case'/'sensor_data.csv'); df['residual']=df.lt101_measured_pct-df.true_level_pct


## 2. Residual → probability mapping – explicit modellfeltevés

Oktatási demonstrációban sigmoid mappinget használunk: nagy negatív residual → magasabb sensor-fault probability.

**Ez nem univerzális fizikai törvény.** A mappinget valós alkalmazásban kalibrálni, validálni és uncertainty-vel ellátni kell.


In [ ]:
def logistic(x): return 1/(1+np.exp(-x))
def residual_to_p(r,center=-2.5,scale=.8,p_min=.002,p_max=.6):
 score=logistic(-(r-center)/scale); return p_min+(p_max-p_min)*score
df['p_sensor']=residual_to_p(df.residual)
plt.plot(df.time_min,df.p_sensor); plt.xlabel('time [min]'); plt.ylabel('P(sensor fault | evidence)'); plt.grid(alpha=.3); plt.show()


## 3. Kontextus: maintenance overdue

Oktatási modellben overdue állapot növelheti a prior oddsot. Ezt szintén explicit modellfeltevésként kezeljük, nem automatikus oksági igazságként.


In [ ]:
overdue=np.asarray(df.maintenance_overdue,dtype=bool)
odds=df.p_sensor/(1-df.p_sensor); odds_adj=odds*np.where(overdue,2.0,1.0); df['p_sensor_context']=odds_adj/(1+odds_adj)
plt.plot(df.time_min,df.p_sensor,label='evidence only'); plt.plot(df.time_min,df.p_sensor_context,label='+ overdue context'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 4. Dynamic FTA

A statikus `LT101_LOW` probability helyére időfüggő posterior kerül. A többi input itt fix marad, így láthatóvá válik az evidence contribution.


In [ ]:
p_ch=.08; p_lshh=.01; p_xv=.005; p_op=.04; p_auto=1-(1-p_lshh)*(1-p_xv)
def p_overflow(p_sensor): return p_ch*p_auto*(1-(1-p_sensor)*(1-p_op))
df['p_overflow']=p_overflow(df.p_sensor_context)
plt.semilogy(df.time_min,df.p_overflow); plt.xlabel('time [min]'); plt.ylabel('dynamic P(overflow)'); plt.grid(alpha=.3); plt.show()


## 5. Risk = probability × consequence

Ha a consequence proxy 50 M HUF, a várható veszteség idősora csak monetáris döntési dimenzió. Safety constraints ettől függetlenül is előírhatnak azonnali beavatkozást.


In [ ]:
C=50_000_000; df['expected_consequence']=df.p_overflow*C
plt.plot(df.time_min,df.expected_consequence); plt.ylabel('expected consequence proxy [HUF]'); plt.xlabel('time [min]'); plt.grid(alpha=.3); plt.show()


## 6. Maintenance trigger

Maintain now vs next opportunity. A döntési küszöb abból adódik, mikor lesz a várakozás expected costja nagyobb.


In [ ]:
Cm=250_000; prod_now=800_000; prod_window=100_000; p_after=.001
cost_now=Cm+prod_now+p_after*C
cost_wait=Cm+prod_window+df.p_overflow*C
df['maintain_now_cheaper']=cost_now<cost_wait
first=df.loc[df.maintain_now_cheaper,'time_min'].min(); print('first economic trigger [min]=',first,'cost now=',cost_now)


In [ ]:
plt.plot(df.time_min,cost_wait,label='wait'); plt.axhline(cost_now,ls='--',label='maintain now'); plt.xlabel('time [min]'); plt.ylabel('expected cost [HUF]'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 7. Döntési lánc és traceability

`sensor data → residual → diagnostic model → posterior fault probability → FTA/Markov risk → consequence model → maintenance decision`.

Minden nyílnál dokumentálni kell: modellverzió, kalibrációs adat, időbélyeg, uncertainty, assumption és responsible component.


In [ ]:
pd.DataFrame([
 {'step':'sensor→residual','assumption':'reference level meaningful'},
 {'step':'residual→P(fault)','assumption':'calibrated evidence mapping'},
 {'step':'P(fault)→FTA','assumption':'static gate structure valid'},
 {'step':'risk→decision','assumption':'cost proxy and constraints valid'}])


## 8. Amit nem szabad automatikusan állítani

ARIMA/ML forecast threshold-crossing probability nem válik automatikusan FMEA occurrence pontszámmá vagy Markov transition rate-té. Ehhez külön kalibrációs modell kell. Ugyanez igaz residual score → failure probability mappingre.


## Próbáld ki!

1. Változtasd a sigmoid center/scale paramétereit és figyeld a risk trendet.
2. Kapcsold ki az overdue odds multipliert.
3. Duplázd consequence értékét és keresd az új economic trigger időt.
4. Tervezz validációs kísérletet a residual→P(fault) mapping kalibrálására.


## Összefoglalás

A dinamikus risk pipeline a diagnosztikai evidenciát döntéstámogató kockázattá alakítja, de minden átalakítás modellfeltevés. A következő integrált tanegységben a teljes PUNDIT-oktatási láncot egyetlen workflow-ban járjuk végig.
